# 4 — Broad lineage: ordered residual gating (5 compartments + Other)

> ## ⚠️ BLOCKED at Gate 2 (2026-07-23) — this notebook has no input yet
>
> Every cell here needs RESTORE `_pos` calls, and **none exist**: no marker pair is accepted, no
> divisor is emitted, and `data/` holds no gated parquet. The composition-trace cell below is
> **retired** (it imported the deleted `MARKER_PAIRS`); `run_lineage` and the post-run validation
> will fail on missing `cfg.broad_dir`. See `docs/restore_faithful_rebuild_plan.md`.
>
> The gating code itself is current and reflects two 2026-07-23 corrections: **CD20 is not an Immune
> anchor** (B cells reach Immune via `CD79a`, since a lone CD20 call on an abundant cell would
> otherwise win an any-positive union), and in the immune sub-split **B/Plasma gate at the LOWEST
> precedence**, so DC / Macrophage / Neutrophil / NK / T claim their cells first — a CD20⁺CD68⁺ cell
> is a Macrophage. Both are dormant until Gate 4 wires production.

Types every cell by the **first matching gate in priority order**, each gate running on the RESIDUAL of the
prior: **Epithelial** (`E_cadherin`; hormone⁺ sub-branch = **Endocrine** β/α/δ) → **Endothelial** (`CD31`)
→ **Neural** (`B3TUBB`) → **Immune** (marker union) → **Mesenchymal** (`SMA` then `Vimentin`) → **Other**
(failed every gate — real panel gaps, NOT force-assigned). `_pos` comes straight from RESTORE (no `_norm`
floor). **Begin with the composition trace**, then run the assignment.

In [ ]:
# Parameters for this step (self-contained -- no config.ini). Edit the paths for your machine.
%load_ext autoreload
%autoreload 2                                         # pick up edits to the phenocycler package without a kernel restart
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `phenocycler` importable from notebooks/
from phenocycler import PipelineConfig

REPO = pathlib.Path.cwd().resolve().parents[1]        # Islet-Explorer-Senior (parent of the submodule; data/ lives here)

# Run on a subset of donors (None = every donor under data/cells/donor_id=*).
DONORS = None            # e.g. ["6374", "6380"] to iterate on a few

# Composition-trace diagnostic knobs (recomputes gate thresholds like RESTORE, on the trace donor).
THRESHOLD_STAT = "mean3sd"
MODEL          = "SSC"
IDX_FLOOR_Q    = 0.5
SUBSAMPLE      = 8000
SEED           = 0

cfg = PipelineConfig(
    data_dir       = REPO / "data",
    donor_metadata = pathlib.Path("/home/smith6jt/IO60panc2nd/donor_metadata_panc.xlsx"),   # disease-status for the composition validation
    n_jobs = 8,                 # per-donor pool for the vectorized ordered-residual gating
)
donors = DONORS or cfg.discover_donors()
print(f"donors: {len(donors)} " + ("(subset)" if DONORS else "(all)") + f" -> {donors[:6]}" + (" ..." if len(donors) > 6 else ""))

## Diagnostics — threshold → composition trace (begin here)

The payoff of the RESTORE thresholds: the `_pos` calls feed the ordered-residual gating tree. Here we
threshold an *illustrative subset* of gate markers on the trace donor with the notebook's current settings
(so changing the floor/statistic flows all the way to composition), assign compartments in priority order,
and split hormone⁺ Epithelial into Endocrine. ~12 SSC fits/donor ≈ a few minutes; the real pipeline (below)
runs every gate marker + the robust guard cohort-wide.

In [ ]:
# RETIRED 2026-07-23 — this diagnostic cannot be repaired before Gate 4, for two reasons.
#
# 1. It imported `MARKER_PAIRS` (deleted) and built `REFMAP = {t: r for t, r in MARKER_PAIRS}`, i.e.
#    ONE reference per target. That map no longer exists and has no faithful replacement: the
#    validation screen evaluates each target against SEVERAL candidate references (CD20 alone has 7
#    — CD3e, CD68, CD163, E_cadherin, EpCAM, Ker8_18, Pan_Cytokeratin), and NONE of them is accepted
#    yet. Picking one here would manufacture a threshold the method has not sanctioned.
#
# 2. Its gate list was stale in the same direction as the pair web:
#       was:  "Immune": ["CD3e", "CD20", "CD68", "CD163"]     # "illustrative subset"
#    CD20 is no longer an Immune anchor at all — the gate is an any-positive union, so a lone CD20
#    call (B cells are sparse in pancreas) could pull an abundant cell into Immune before a more
#    frequent compartment claimed it. B cells reach Immune via CD79a and are sub-typed there, where
#    B/Plasma now gate at the LOWEST precedence. The real gate is:
#       COMPARTMENT_GATES["Immune"] = ["CD3e", "CD79a", "CD68", "CD163", "CD206",
#                                      "Iba1", "CD11b", "CD11c", "MPO"]
#    (phenocycler/config.py) — read it from there rather than re-typing a subset.
#
# It also recomputed thresholds with the retired `mean3sd` statistic. The manuscript-faithful method
# uses an NNMF separator and a divisor equal to the MAXIMUM target intensity among the accepted
# target-negative controls. See docs/restore_faithful_rebuild_plan.md; ground truth is RESTORE.pdf.
#
# Gate 2 is a human pair review, not a recomputation:
#   data/restore_pair_validation/expanded_review_v10/START_HERE_RESTORE_REVIEW.txt
# Restore the original cell with: git show 09a2069^:notebooks/04_broad_lineage.ipynb
raise NotImplementedError(
    "composition trace retired 2026-07-23: MARKER_PAIRS is deleted and no reference is accepted yet; "
    "record the Gate-2 pair review in data/restore_pair_validation/expanded_review_v10/."
)

## Run — broad lineage on the (subset of) donors

In [ ]:
from phenocycler.lineage import run_lineage
C = run_lineage(cfg, donors=DONORS, n_jobs=cfg.n_jobs)   # donors=None -> all
C

## Post-run validation

Cells typed, the `Other` fraction (real panel gaps, not force-assigned), per-compartment counts, and
composition (% of cells) by disease status — read directly from the `compartment` column. Endocrine should
fall and Immune rise ND → Aab+ → T1D.

In [ ]:
from collections import Counter
import pandas as pd
import pyarrow.dataset as ds
from phenocycler.lineage import status_map
from phenocycler.config import COMPARTMENT_ORDER, OTHER_LABEL, STATUS_ORDER

_ORDER = COMPARTMENT_ORDER + [OTHER_LABEL]
smap = status_map(cfg)
total, other = 0, 0
comp_counts, status_comp = Counter(), {}
for d in donors:
    t = ds.dataset(cfg.broad_dir / f"donor_id={d}", format="parquet").to_table(columns=["compartment"]).to_pandas()
    total += len(t); other += int((t["compartment"] == OTHER_LABEL).sum())
    vc = t["compartment"].value_counts().to_dict()
    comp_counts.update(vc)
    status_comp.setdefault(smap.get(d, "?"), Counter()).update(vc)
print(f"Total cells typed: {total:,}")
print(f"{OTHER_LABEL} (failed every gate, real panel gaps): {100*other/total:.1f}%\n")
vcs = pd.Series(comp_counts).reindex(_ORDER).fillna(0).astype(int)
display(vcs.to_frame("cells").assign(**{"%": (100 * vcs / total).round(2)}))
if smap:
    comp = pd.DataFrame(status_comp).T.fillna(0)
    comp = comp.div(comp.sum(axis=1), axis=0).mul(100).round(1)
    order = [s for s in STATUS_ORDER if s in comp.index] + [s for s in comp.index if s not in STATUS_ORDER]
    comp = comp.reindex(index=order, columns=[c for c in _ORDER if c in comp.columns])
    print("Composition (% of cells) by disease status:")
    display(comp)

In [ ]:
from IPython.display import Image, display
fig = cfg.phenotype_dir / "broad_lineage_composition.png"
display(Image(filename=str(fig))) if fig.exists() else print(f"({fig.name} not found — run the lineage step)")